In [14]:
import subprocess
import sys
import os
import time
from datetime import datetime

# 定义颜色代码以便在终端中清晰显示状态
class Colors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'

def log(message, level="INFO"):
    """简单的日志打印函数"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if level == "INFO":
        print(f"[{timestamp}] {Colors.OKBLUE}[INFO]{Colors.ENDC} {message}")
    elif level == "SUCCESS":
        print(f"[{timestamp}] {Colors.OKGREEN}[SUCCESS]{Colors.ENDC} {message}")
    elif level == "ERROR":
        print(f"[{timestamp}] {Colors.FAIL}[ERROR]{Colors.ENDC} {message}")

def run_step(script_name, arg1, arg2, step_description):
    """
    执行单个Benchmark步骤的通用函数
    
    Args:
        script_name (str): bash脚本名称 (e.g., benchmark_ligandmpnn.sh)
        arg1 (str): 第一个参数 (e.g., 输入文件夹路径)
        arg2 (str): 第二个参数 (e.g., 输出文件夹或模式名称)
        step_description (str): 步骤描述，用于日志
    """
    log(f"开始执行步骤: {step_description} ...")
    log(f"指令: bash {script_name} {arg1} {arg2}")

    # 检查脚本是否存在
    if not os.path.exists(script_name):
        log(f"找不到脚本文件: {script_name}", "ERROR")
        sys.exit(1)

    start_time = time.time()
    
    try:
        # 使用 subprocess.run 执行 bash 命令
        # check=True 会在脚本返回非零状态码时抛出异常
        result = subprocess.run(
            ["bash", script_name, arg1, arg2],
            check=True,
            text=True,
            capture_output=False # 如果设为True，则不会实时打印子脚本输出
        )
        
        elapsed_time = time.time() - start_time
        log(f"步骤 '{step_description}' 完成。耗时: {elapsed_time:.2f} 秒", "SUCCESS")
        
    except subprocess.CalledProcessError as e:
        log(f"步骤 '{step_description}' 执行失败！退出代码: {e.returncode}", "ERROR")
        sys.exit(1)
    except Exception as e:
        log(f"发生未预期的错误: {str(e)}", "ERROR")
        sys.exit(1)

def main():
    log("=== 启动多肽设计 Benchmark Pipeline ===")
    
    # --- 步骤 1: LigandMPNN 标准设计 ---
    # 输入: Merged_PDBs
    # 模式: LigandMPNN
    run_step(
        script_name="benchmark_ligandmpnn.sh",
        arg1="datasets/PepSet-noise-0.5/fix_pdb.json",
        arg2="outputs/PepSet-noise-0.5/LigandMPNN",
        step_description="LigandMPNN Design (Standard)"
    )
    
    print("-" * 50)

    # --- 步骤 2: LigandMPNN-HETATM 设计 ---
    # 专家注：通常用于处理含非天然氨基酸或小分子辅因子的复杂体系
    # 输入: Processed_PDBs (注意输入源变更)
    # 模式: LigandMPNN-HETATM
    run_step(
        script_name="benchmark_ligandmpnn.sh",
        arg1="datasets/PepSet-noise-0.5/Processed-fix_pdb.json",
        arg2="outputs/PepSet-noise-0.5/LigandMPNN-HETATM",
        step_description="LigandMPNN Design (Explicit HETATM)"
    )

    print("-" * 50)

    # --- 步骤 3: ProteinMPNN 设计 ---
    # 专家注：作为Baseline或仅蛋白骨架设计的对照
    # 输入: Merged_PDBs
    # 模式: ProteinMPNN
    run_step(
        script_name="benchmark_proteinmpnn.sh",
        arg1="datasets/PepSet-noise-0.5/fix_pdb.json",
        arg2="outputs/PepSet-noise-0.5/ProteinMPNN",
        step_description="ProteinMPNN Design (Baseline)"
    )

    print("-" * 50)
    log("=== 所有 Benchmark 步骤均已成功完成 ===", "SUCCESS")

if __name__ == "__main__":
    main()




[2026-03-18 21:59:38] [INFO] === 启动多肽设计 Benchmark Pipeline ===
[2026-03-18 21:59:38] [INFO] 开始执行步骤: LigandMPNN Design (Standard) ...
[2026-03-18 21:59:38] [INFO] 指令: bash benchmark_ligandmpnn.sh datasets/PepSet-noise-0.5/fix_pdb.json outputs/PepSet-noise-0.5/LigandMPNN
Running LigandMPNN with model: ligandmpnn_v_32_005_25
Designing protein from this path: /home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/test_weight/datasets/PepSet-noise-0.5/Merged_PDBs/4zoz.pdb
These residues will be redesigned:  ['L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8', 'L9', 'L10', 'L11', 'L12', 'L13']
These residues will be fixed:  ['A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63', 'A64', 'A65', 'A66', 'A67', 'A68', 'A69', 'A70', 'A71', 'A72', 'A73', 'A74', 'A75', 'A76', 'A77', 'A78', 'A79', 'A80', 'A81', 'A82', 'A83', 'A84', 'A85', 'A86', 'A87', 'A88', 'A89', 'A90', 'A91', 'A92', 'A93', 'A94', 'A95', 'A96', 'A97', 'A98', 'A99', 'A100', 'A101', 'A102', 'A103', 'A128', 'A129', 'A130'